In [ ]:
# clean_dataフォルダのclean_data.csvは入力がゲームのプレイログ、出力は興奮度で、教師あり学習の形が既にできている
# このコードはデータ分析

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CLEAN_PATH = Path("../clean_data/clean_data.csv")
OUTLIER_PATH = Path("../clean_data/outliers.csv")
OUT_DIR = Path("../out/again_analysis")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CLEAN_PATH)
outliers = pd.read_csv(OUTLIER_PATH)

print("=== Basic Info ===")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Genres:", df["[control]genre"].unique())
print("Games:", df["[control]game"].unique())
print("Players:", df["[control]player_id"].nunique())
print("Sessions:", df["[control]session_id"].nunique())

print("\n=== Outliers ===")
print("Outlier sessions:", len(outliers))
print(outliers["[control]game"].value_counts())

# -------------------------
# 1. セッションごとのデータ数
# -------------------------
session_counts = df.groupby("[control]session_id").size()

print("\n=== Session Length ===")
print(session_counts.describe())

plt.figure()
session_counts.hist(bins=30)
plt.title("Number of rows per session")
plt.xlabel("Rows")
plt.ylabel("Count")
plt.savefig(OUT_DIR / "session_length_hist.png")
plt.close()

# -------------------------
# 2. arousal の分布
# -------------------------
target = "[output]arousal"

print("\n=== Arousal ===")
print(df[target].describe())

plt.figure()
df[target].hist(bins=50)
plt.title("Arousal distribution")
plt.xlabel("Arousal")
plt.ylabel("Count")
plt.savefig(OUT_DIR / "arousal_distribution.png")
plt.close()

# ゲーム別 arousal
plt.figure()
df.boxplot(column=target, by="[control]game", rot=45)
plt.title("Arousal by game")
plt.suptitle("")
plt.xlabel("Game")
plt.ylabel("Arousal")
plt.tight_layout()
plt.savefig(OUT_DIR / "arousal_by_game.png")
plt.close()

# -------------------------
# 3. 欠損率
# -------------------------
missing_rate = df.isna().mean().sort_values(ascending=False)
missing_rate.to_csv(OUT_DIR / "missing_rate.csv")

print("\n=== Missing rate top 20 ===")
print(missing_rate.head(20))

plt.figure(figsize=(8, 10))
missing_rate.head(40).plot(kind="barh")
plt.title("Top 40 missing-rate columns")
plt.xlabel("Missing rate")
plt.tight_layout()
plt.savefig(OUT_DIR / "missing_rate_top40.png")
plt.close()

# -------------------------
# 4. 数値列だけ抽出
# -------------------------
control_cols = [c for c in df.columns if c.startswith("[control]")]
string_cols = [c for c in df.columns if c.startswith("[string]")]
output_cols = [c for c in df.columns if c.startswith("[output]")]

feature_cols = [
    c for c in df.columns
    if c not in control_cols + string_cols + output_cols
]

numeric_df = df[feature_cols].apply(pd.to_numeric, errors="coerce")

print("\n=== Numeric feature candidates ===")
print("Candidate features:", len(numeric_df.columns))

# 欠損が多すぎる列を確認
numeric_missing = numeric_df.isna().mean().sort_values(ascending=False)
numeric_missing.to_csv(OUT_DIR / "numeric_missing_rate.csv")

# -------------------------
# 5. 分散がほぼない列
# -------------------------
variance = numeric_df.var(numeric_only=True).sort_values()
variance.to_csv(OUT_DIR / "feature_variance.csv")

low_variance_cols = variance[variance < 1e-8].index.tolist()

print("\n=== Low variance columns ===")
print("Low variance columns:", len(low_variance_cols))
print(low_variance_cols[:30])

# -------------------------
# 6. arousalとの相関
# -------------------------
corr_df = pd.concat([numeric_df, df[target]], axis=1)
corr = corr_df.corr(numeric_only=True)[target].drop(target)
corr_abs = corr.abs().sort_values(ascending=False)

corr_abs.to_csv(OUT_DIR / "arousal_correlation_abs.csv")
corr.sort_values(ascending=False).to_csv(OUT_DIR / "arousal_correlation_signed.csv")

print("\n=== Top correlated features with arousal ===")
print(corr_abs.head(30))

plt.figure(figsize=(8, 10))
corr_abs.head(30).sort_values().plot(kind="barh")
plt.title("Top 30 features correlated with arousal")
plt.xlabel("Absolute correlation")
plt.tight_layout()
plt.savefig(OUT_DIR / "arousal_correlation_top30.png")
plt.close()

# -------------------------
# 7. ゲーム別に使われている特徴量
# -------------------------
game_feature_usage = {}

for game, gdf in df.groupby("[control]game"):
    gnum = gdf[feature_cols].apply(pd.to_numeric, errors="coerce")
    usage = {
        "non_missing_rate": 1 - gnum.isna().mean(),
        "non_zero_rate": (gnum.fillna(0) != 0).mean(),
        "variance": gnum.var(numeric_only=True),
    }
    game_feature_usage[game] = pd.DataFrame(usage)

usage_all = pd.concat(game_feature_usage, names=["game", "feature"])
usage_all.to_csv(OUT_DIR / "game_feature_usage.csv")

print("\n=== Saved game feature usage ===")
print(OUT_DIR / "game_feature_usage.csv")

# -------------------------
# 8. 自作RPGに転用しやすそうな汎用特徴量
# -------------------------
rpg_like_features = [
    "[general]input_intensity",
    "[general]input_diversity",
    "[general]activity",
    "[general]score",
    "[general]bot_count",
    "[general]bot_diversity",
    "[general]bot_movement",
    "[general]player_movement",
    "[general]object_intensity",
    "[general]object_diversity",
    "[general]event_intensity",
    "[general]event_diversity",
    "key_press_count",
    "idle_time",
    "player_score",
    "player_kill_count",
    "player_health",
    "player_damaged",
    "player_death",
    "visible_bot_count",
    "bot_health",
    "bot_damaged",
    "bot_player_distance",
]

rpg_like_features = [c for c in rpg_like_features if c in df.columns]

rpg_df = df[rpg_like_features + [target]].apply(pd.to_numeric, errors="coerce")
rpg_corr = rpg_df.corr(numeric_only=True)[target].drop(target).abs().sort_values(ascending=False)

print("\n=== RPG-like features correlation ===")
print(rpg_corr)

rpg_corr.to_csv(OUT_DIR / "rpg_like_feature_correlation.csv")

plt.figure(figsize=(8, 8))
rpg_corr.sort_values().plot(kind="barh")
plt.title("RPG-like features correlated with arousal")
plt.xlabel("Absolute correlation")
plt.tight_layout()
plt.savefig(OUT_DIR / "rpg_like_feature_correlation.png")
plt.close()

# -------------------------
# 9. セッション単位のarousal変化を可視化
# -------------------------
sample_sessions = df["[control]session_id"].drop_duplicates().head(5)

for sid in sample_sessions:
    sdf = df[df["[control]session_id"] == sid].copy()
    sdf["time"] = pd.to_numeric(sdf["[control]time_stamp"], errors="coerce")

    plt.figure()
    plt.plot(sdf["time"], sdf[target])
    game = sdf["[control]game"].iloc[0]
    plt.title(f"Arousal trace: {game} / {sid[:8]}")
    plt.xlabel("Time stamp")
    plt.ylabel("Arousal")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"arousal_trace_{sid[:8]}.png")
    plt.close()

print("\nAnalysis finished.")
print("Results saved to:", OUT_DIR)

C:\Users\tatsu\AppData\Local\Temp\ipykernel_48528\163840148.py:11: DtypeWarning: Columns (23,24,98,99,100) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CLEAN_PATH)


=== Basic Info ===
Rows: 487176
Columns: 121
Genres: ['Platformer' 'Racing' 'Shooter']
Games: ['Endless' 'Pirates!' "Run'N'Gun" 'ApexSpeed' 'Solid' 'TinyCars' 'Heist!'
 'Shootout' 'TopDown']
Players: 122
Sessions: 995

=== Outliers ===
Outlier sessions: 121
[control]game
Shootout     18
Solid        15
TinyCars     15
Pirates!     14
Run'N'Gun    14
Heist!       14
Endless      12
ApexSpeed    10
TopDown       9
Name: count, dtype: int64

=== Session Length ===
count    995.000000
mean     489.624121
std       17.643788
min      285.000000
25%      490.000000
50%      492.000000
75%      492.000000
max      522.000000
dtype: float64

=== Arousal ===
count    487176.000000
mean          0.464794
std           0.290209
min           0.000000
25%           0.221311
50%           0.453839
75%           0.700000
max           1.000000
Name: [output]arousal, dtype: float64

=== Missing rate top 20 ===
[string]player_damaged_by       0.994579
[string]bot_damaged_by          0.972022
[string]p

<Figure size 640x480 with 0 Axes>

In [2]:
OUT_DIR = Path("../out/again_analysis_rpg_features")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CLEAN_PATH, low_memory=False)

GAME_COL = "[control]game"
TARGET = "[output]arousal"

candidate_features = [
    "[general]input_intensity",
    "[general]input_diversity",
    "[general]activity",
    "[general]score",
    "[general]event_intensity",
    "[general]event_diversity",
    "[general]bot_count",
    "[general]bot_diversity",
    "[general]bot_movement",
    "[general]player_movement",
    "[general]object_intensity",
    "[general]object_diversity",

    "key_press_count",
    "idle_time",
    "player_score",
    "player_health",
    "player_damaged",
    "player_death",
    "player_kill_count",
    "player_movement" if "player_movement" in df.columns else None,

    "visible_bot_count",
    "bot_health",
    "bot_damaged",
    "bot_player_distance",
]

candidate_features = [
    c for c in candidate_features
    if c is not None and c in df.columns
]

print("Candidate features:")
for c in candidate_features:
    print("-", c)

# 数値化
for col in candidate_features + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

games = sorted(df[GAME_COL].dropna().unique())

# =====================================
# 1. ゲーム別 欠損率
# =====================================
missing_rate = pd.DataFrame(index=candidate_features, columns=games)

for game in games:
    gdf = df[df[GAME_COL] == game]
    missing_rate[game] = gdf[candidate_features].isna().mean()

missing_rate.to_csv(OUT_DIR / "game_missing_rate.csv", encoding="utf-8-sig")

# =====================================
# 2. ゲーム別 非ゼロ率
# =====================================
nonzero_rate = pd.DataFrame(index=candidate_features, columns=games)

for game in games:
    gdf = df[df[GAME_COL] == game]
    nonzero_rate[game] = (gdf[candidate_features].fillna(0) != 0).mean()

nonzero_rate.to_csv(OUT_DIR / "game_nonzero_rate.csv", encoding="utf-8-sig")

# =====================================
# 3. ゲーム別 arousal 相関
# =====================================
corr_by_game = pd.DataFrame(index=candidate_features, columns=games)

for game in games:
    gdf = df[df[GAME_COL] == game]

    for feature in candidate_features:
        x = gdf[feature]
        y = gdf[TARGET]

        valid = x.notna() & y.notna()

        if valid.sum() < 10:
            corr = np.nan
        elif x[valid].nunique() <= 1:
            corr = np.nan
        else:
            corr = x[valid].corr(y[valid], method="pearson")

        corr_by_game.loc[feature, game] = corr

corr_by_game = corr_by_game.astype(float)
corr_by_game.to_csv(OUT_DIR / "game_arousal_correlation.csv", encoding="utf-8-sig")

# 絶対値版
corr_abs_by_game = corr_by_game.abs()
corr_abs_by_game.to_csv(OUT_DIR / "game_arousal_correlation_abs.csv", encoding="utf-8-sig")

# =====================================
# 4. 発表用：特徴量ごとの総合スコア
# =====================================
summary = pd.DataFrame(index=candidate_features)

summary["mean_missing_rate"] = missing_rate.astype(float).mean(axis=1)
summary["mean_nonzero_rate"] = nonzero_rate.astype(float).mean(axis=1)
summary["mean_abs_corr"] = corr_abs_by_game.mean(axis=1)
summary["max_abs_corr"] = corr_abs_by_game.max(axis=1)
summary["valid_game_count"] = corr_abs_by_game.notna().sum(axis=1)

summary["recommend_score"] = (
    (1 - summary["mean_missing_rate"]) * 0.3
    + summary["mean_nonzero_rate"] * 0.3
    + summary["mean_abs_corr"].fillna(0) * 0.4
)

summary = summary.sort_values("recommend_score", ascending=False)
summary.to_csv(OUT_DIR / "rpg_feature_summary.csv", encoding="utf-8-sig")

print("\n=== RPG feature summary ===")
print(summary)

# =====================================
# 5. ヒートマップ描画関数
# =====================================
def save_heatmap(data, title, filename, vmin=None, vmax=None):
    plt.figure(figsize=(12, max(6, len(data) * 0.35)))
    plt.imshow(data.astype(float), aspect="auto", vmin=vmin, vmax=vmax)

    plt.colorbar()
    plt.xticks(range(len(data.columns)), data.columns, rotation=45, ha="right")
    plt.yticks(range(len(data.index)), data.index)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=200)
    plt.close()

save_heatmap(
    missing_rate.astype(float),
    "Missing rate by game",
    "heatmap_missing_rate.png",
    vmin=0,
    vmax=1
)

save_heatmap(
    nonzero_rate.astype(float),
    "Non-zero rate by game",
    "heatmap_nonzero_rate.png",
    vmin=0,
    vmax=1
)

save_heatmap(
    corr_abs_by_game.astype(float),
    "Absolute correlation with arousal by game",
    "heatmap_arousal_correlation_abs.png",
    vmin=0,
    vmax=0.5
)

# =====================================
# 6. ゲーム別に相関上位を出力
# =====================================
top_corr_rows = []

for game in games:
    top = corr_abs_by_game[game].dropna().sort_values(ascending=False).head(10)

    for feature, value in top.items():
        signed_corr = corr_by_game.loc[feature, game]
        top_corr_rows.append({
            "game": game,
            "feature": feature,
            "abs_corr": value,
            "signed_corr": signed_corr,
            "missing_rate": missing_rate.loc[feature, game],
            "nonzero_rate": nonzero_rate.loc[feature, game],
        })

top_corr_df = pd.DataFrame(top_corr_rows)
top_corr_df.to_csv(OUT_DIR / "top10_features_by_game.csv", index=False, encoding="utf-8-sig")

print("\nSaved results to:", OUT_DIR)

Candidate features:
- [general]input_intensity
- [general]input_diversity
- [general]activity
- [general]score
- [general]event_intensity
- [general]event_diversity
- [general]bot_count
- [general]bot_diversity
- [general]bot_movement
- [general]player_movement
- [general]object_intensity
- [general]object_diversity
- key_press_count
- idle_time
- player_score
- player_health
- player_damaged
- player_death
- player_kill_count
- visible_bot_count
- bot_health
- bot_damaged
- bot_player_distance

=== RPG feature summary ===
                           mean_missing_rate  mean_nonzero_rate  \
[general]score                      0.000000           0.911072   
player_score                        0.000000           0.911072   
[general]player_movement            0.000000           0.933220   
visible_bot_count                   0.000000           0.589049   
[general]bot_count                  0.000000           0.589049   
[general]activity                   0.000000           0.708720   
[g

In [ ]:
# =====================================
# 7. 特徴量 vs arousal の散布図を1つずつ表示
# =====================================

# 表示したい特徴量
plot_features = candidate_features

# データ数が多すぎるので、散布図用にサンプリングする
# 全点を描くと重すぎるため
MAX_POINTS = 3000

for game in games:
    gdf = df[df[GAME_COL] == game].copy()

    print(f"\n===== {game} =====")

    for feature in plot_features:
        x = pd.to_numeric(gdf[feature], errors="coerce")
        y = pd.to_numeric(gdf[TARGET], errors="coerce")

        valid = x.notna() & y.notna()

        # データが少ない、または値が全部同じならスキップ
        if valid.sum() < 30:
            print(f"skip: {feature} / valid data too small")
            continue

        if x[valid].nunique() <= 1:
            print(f"skip: {feature} / feature has only one value")
            continue

        plot_df = pd.DataFrame({
            "feature": x[valid],
            "arousal": y[valid],
        })

        # 点が多すぎる場合はランダムサンプリング
        if len(plot_df) > MAX_POINTS:
            plot_df = plot_df.sample(MAX_POINTS, random_state=42)

        corr = plot_df["feature"].corr(plot_df["arousal"], method="pearson")

        plt.figure(figsize=(6, 4))
        plt.scatter(
            plot_df["feature"],
            plot_df["arousal"],
            alpha=0.25,
            s=8
        )

        plt.title(f"{game}: {feature}\nPearson r = {corr:.3f}")
        plt.xlabel(feature)
        plt.ylabel("arousal")
        plt.grid(True)
        plt.tight_layout()
        plt.show()